<a href="https://colab.research.google.com/github/AstronomekDomansky/-.-1/blob/main/%D0%A1%D0%B8%D0%B8%D0%91%D0%941.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [206]:
# ==========================================================
# ⬇️  ЗАПОЛНИТЕ ЭТИ ТРИ ПОЛЯ СВОИМИ ДАННЫМИ
# ==========================================================
last_name   = "Арутюнян"    # <-- впишите свою фамилию
first_name  = "Дмитрий"      # <-- впишите своё имя
list_number = 2           # <-- ваш номер по списку (целое число)
# ==========================================================

import hashlib

def make_seed(last: str, first: str, number: int) -> int:
    # ФИО + номер -> стабильный seed 0..99999
    key = f"{last.strip().lower()}|{first.strip().lower()}|{int(number)}"
    return int(hashlib.sha256(key.encode("utf-8")).hexdigest(), 16) % 100_000

SEED = make_seed(last_name, first_name, list_number)

print("Студент:", last_name, first_name, f"(№{list_number})")
print("Ваш персональный SEED:", SEED)
print("Запишите SEED в итоговый отчёт — он подтверждает вариант.")


Студент: Арутюнян Дмитрий (№2)
Ваш персональный SEED: 36390
Запишите SEED в итоговый отчёт — он подтверждает вариант.


In [207]:
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 50)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 4)


In [208]:
#@title Создание персонального датасета (нажмите ▶️) { display-mode: "form" }

def load_it_tickets(seed: int, n: int = 1800) -> "pd.DataFrame":
    """Индивидуальный «грязный» журнал IT-обращений по seed."""
    rng = np.random.default_rng(seed)

    opened_ts = rng.integers(1_735_689_600, 1_751_241_600, size=n)
    department = rng.choice(
        ["Support", "Backend", "DevOps", "Security", "Data"],
        size=n, p=[0.32, 0.22, 0.18, 0.14, 0.14],
    )
    product = rng.choice(
        ["Cloud", "CRM", "Billing", "Auth", "Analytics"], size=n
    )
    priority = rng.choice(
        ["Low", "Medium", "High", "Critical"], size=n, p=[0.40, 0.30, 0.20, 0.10]
    )
    channel = rng.choice(["Email", "Slack", "Portal", "Phone"], size=n)
    assignee = rng.choice(["Junior", "Middle", "Senior"], size=n, p=[0.35, 0.45, 0.20])
    tier = rng.choice(["Free", "Pro", "Enterprise"], size=n, p=[0.45, 0.40, 0.15])
    cpu_pct = np.clip(np.round(rng.normal(48, 18, size=n), 1), 1, 99)
    memory_mb = np.clip(np.round(rng.normal(2200, 850, size=n), 0), 256, None)
    n_reopens = rng.poisson(0.7, size=n)
    comments = rng.poisson(4.0, size=n) + 1
    resolve_hours = np.clip(np.round(rng.lognormal(1.55, 0.75, size=n), 2), 0.2, 240)

    logit = (
        -1.4
        + 1.4 * (priority == "Critical")
        + 0.7 * (priority == "High")
        - 0.4 * (priority == "Low")
        + 0.035 * resolve_hours
        + 0.018 * cpu_pct
        + 0.55 * (department == "Security")
        + 0.35 * (n_reopens >= 2)
        - 0.45 * (assignee == "Senior")
        - 0.30 * (tier == "Enterprise")
    )
    sla = (rng.uniform(size=n) < 1 / (1 + np.exp(-logit))).astype(int)

    df = pd.DataFrame({
        "ticket_id": np.arange(10_000, 10_000 + n),
        "opened_ts": opened_ts,
        "department": department,
        "product": product,
        "priority": priority,
        "channel": channel,
        "assignee_level": assignee,
        "client_tier": tier,
        "cpu_pct": cpu_pct,
        "memory_mb": memory_mb,
        "n_reopens": n_reopens,
        "comments": comments,
        "resolve_hours": resolve_hours,
        "sla_breached": sla,
    })

    df.loc[rng.choice(n, int(0.06 * n), replace=False), "resolve_hours"] = np.nan
    df.loc[rng.choice(n, int(0.04 * n), replace=False), "cpu_pct"] = np.nan
    df.loc[rng.choice(n, int(0.03 * n), replace=False), "comments"] = np.nan
    df.loc[rng.choice(n, 10, replace=False), "resolve_hours"] *= 9
    df.loc[rng.choice(n, 8, replace=False), "memory_mb"] *= 7
    ridx = rng.choice(n, int(0.12 * n), replace=False)
    df.loc[ridx, "department"] = df.loc[ridx, "department"].str.upper()
    pidx = rng.choice(n, int(0.08 * n), replace=False)
    df.loc[pidx, "product"] = "  " + df.loc[pidx, "product"].str.lower() + " "
    df = pd.concat([df, df.sample(int(0.025 * n), random_state=seed)], ignore_index=True)
    return df.sample(frac=1, random_state=seed).reset_index(drop=True)

print("Функция load_it_tickets готова.")


Функция load_it_tickets готова.


In [209]:
df_raw = load_it_tickets(SEED)

print("Ваш датасет создан. Размер:", df_raw.shape)
print("Доля нарушений SLA:", f"{df_raw['sla_breached'].mean():.1%}")


Ваш датасет создан. Размер: (1845, 14)
Доля нарушений SLA: 44.2%


---
# Этап 1. Основы pandas (практика 1)

**Цель.** Убедиться, что таблица загрузилась, понять размер и типы, выбрать строки и столбцы, добавить признак, сохранить фрагмент.

### Пример

Стандартный осмотр и простая выборка. Запустите ячейки — те же команды понадобятся в задании.


In [210]:
print("Размер (строки, столбцы):", df_raw.shape)
print("Имена столбцов:", list(df_raw.columns))
print("\nПервые 5 строк:")
display(df_raw.head())
print("\nТипы и непропуски:")
df_raw.info()


Размер (строки, столбцы): (1845, 14)
Имена столбцов: ['ticket_id', 'opened_ts', 'department', 'product', 'priority', 'channel', 'assignee_level', 'client_tier', 'cpu_pct', 'memory_mb', 'n_reopens', 'comments', 'resolve_hours', 'sla_breached']

Первые 5 строк:


,ticket_id,opened_ts,department,product,priority,channel,assignee_level,client_tier,cpu_pct,memory_mb,n_reopens,comments,resolve_hours,sla_breached
0,11747,1749778657,DevOps,Auth,Low,Phone,Middle,Enterprise,97.0,1982.0,1,5.0,5.74,0
1,10498,1747451354,BACKEND,Analytics,Medium,Email,Middle,Pro,81.6,3871.0,0,NaN,NaN,0
2,11798,1746270023,Support,Analytics,Low,Phone,Junior,Pro,47.6,1569.0,1,8.0,2.37,0
3,10627,1749975882,Backend,CRM,Low,Phone,Junior,Enterprise,40.6,2006.0,1,8.0,1.72,0
4,11516,1745730831,Backend,CRM,Medium,Portal,Junior,Pro,59.4,2678.0,0,7.0,5.38,1



Типы и непропуски:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1845 entries, 0 to 1844
Data columns (total 14 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   ticket_id       1845 non-null   int64  
 1   opened_ts       1845 non-null   int64  
 2   department      1845 non-null   object 
 3   product         1845 non-null   object 
 4   priority        1845 non-null   object 
 5   channel         1845 non-null   object 
 6   assignee_level  1845 non-null   object 
 7   client_tier     1845 non-null   object 
 8   cpu_pct         1771 non-null   float64
 9   memory_mb       1845 non-null   float64
 10  n_reopens       1845 non-null   int64  
 11  comments        1791 non-null   float64
 12  resolve_hours   1736 non-null   float64
 13  sla_breached    1845 non-null   int64  
dtypes: float64(4), int64(4), object(6)
memory usage: 201.9+ KB


In [211]:
# loc — по подписям, iloc — по номерам
print("Один столбец (первые значения priority):")
print(df_raw["priority"].head())
print("\nДве строки и три столбца через loc:")
display(df_raw.loc[0:1, ["ticket_id", "department", "priority"]])
print("Критические тикеты:", (df_raw["priority"] == "Critical").sum())


Один столбец (первые значения priority):
0       Low
1    Medium
2       Low
3       Low
4    Medium
Name: priority, dtype: object

Две строки и три столбца через loc:


,ticket_id,department,priority
0,11747,DevOps,Low
1,10498,BACKEND,Medium


Критические тикеты: 185


In [212]:
# ВАШ КОД (Задание 1)

# 1. Последние 8 строк
print("Последние 8 строк (хвост файла):")
df_raw.tail(8)



Последние 8 строк (хвост файла):


,ticket_id,opened_ts,department,product,priority,channel,assignee_level,client_tier,cpu_pct,memory_mb,n_reopens,comments,resolve_hours,sla_breached
1837,10064,1737413622,Security,Billing,Low,Phone,Junior,Free,37.0,1861.0,1,4.0,13.28,0
1838,11345,1745381224,Backend,Billing,High,Portal,Middle,Free,58.2,3043.0,1,5.0,2.07,1
1839,11103,1740124306,Security,Auth,Critical,Phone,Middle,Free,65.0,2084.0,0,4.0,7.21,1
1840,11440,1743182812,Support,Cloud,Medium,Email,Junior,Free,71.0,2342.0,1,5.0,9.90,1
1841,11468,1737638784,Backend,Billing,Critical,Email,Junior,Enterprise,63.6,459.0,0,6.0,3.17,1
1842,11502,1741010663,SUPPORT,CRM,Medium,Portal,Senior,Free,30.5,1374.0,2,7.0,1.83,0
1843,10777,1739740575,DevOps,CRM,Medium,Email,Junior,Enterprise,47.9,3760.0,0,6.0,9.58,0
1844,10039,1742065231,Data,Auth,Medium,Phone,Middle,Pro,48.5,2117.0,0,5.0,7.72,1


In [213]:
# 2. Число дубликатов
print("Число дубликатов:")
df_raw.duplicated().sum()


Число дубликатов:


np.int64(45)

In [214]:
# 3. High или Critical, выбранные столбцы
print("Количество столбцов:")
subset = df_raw.loc[(df_raw["priority"]=="High" ) | (df_raw["priority"]=="Critical"), ["ticket_id", "department", "priority","resolve_hours"]]
len(subset)


Количество столбцов:


522

In [220]:
# 4. Столбец is_slow и переименование n_reopens → reopen_count
df_raw["is_slow"] = (df_raw["resolve_hours"] > 24).astype(bool)
df_raw[["is_slow", "resolve_hours"]].head(15)

df_raw.rename(columns={"n_reopens" : "reopen_count"}, inplace=True)
df_raw[["is_slow", "reopen_count"]].head(15)



,is_slow,reopen_count
0,False,1
1,False,0
2,False,1
3,False,1
4,False,0
5,False,0
6,False,1
7,False,1
8,False,2
9,False,0


In [223]:
# 5. Сохранение it_tickets_head.csv
df_raw.to_csv("iit_tickets_head.csv", index=False)
print("Сохранено it_tickets_head.csv, строк:", len(df_raw))

# КОНЕЦ ВАШЕГО КОДА


Сохранено it_tickets_head.csv, строк: 1845


**Контрольные вопросы к этапу 1** (ответьте текстом):

1. Чем `.info()` отличается от `.describe()`? Что показывает каждый?
2. Когда удобнее `.loc`, а когда `.iloc`?
3. Почему дубликаты строк могут испортить и статистику, и будущую модель?


*Ответы (двойной клик):*

1.

2.

3.
